In [ ]:
%%sql -r Use_database
Use database anycompany_lab;

# Création du schéma ANALYTICS

In [ ]:
%%sql -r dataframe_1
create SCHEMA if not exists ANALYTICS;

# Table ventes enrichies

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE TABLE ANALYTICS.vente_enrichies AS
SELECT
    f.transaction_id,
    f.transaction_date,
    YEAR(f.transaction_date) AS year,
    MONTH(f.transaction_date) AS month,
    f.amount,
    f.payment_method,
    f.region,

    p.promotion_id,
    p.promotion_type,
    p.discount_percentage,

    m.campaign_id,
    m.campaign_type,
    m.budget,
    m.conversion_rate,

    CASE 
        WHEN p.promotion_id IS NULL THEN 0
        ELSE 1
    END AS promotion_flag,

    CASE 
        WHEN m.campaign_id IS NULL THEN 0
        ELSE 1
    END AS campaign_flag

FROM anycompany_lab.silver.financial_transactions_clean f

LEFT JOIN anycompany_lab.silver.promotions_data_clean p
    ON f.region = p.region
    AND f.transaction_date BETWEEN p.start_date AND p.end_date

LEFT JOIN anycompany_lab.silver.marketing_campaigns_clean m
    ON f.region = m.region
    AND f.transaction_date BETWEEN m.start_date AND m.end_date

WHERE f.transaction_type = 'Sale';

In [ ]:
%%sql -r dataframe_5
SELECT * from vente_enrichies;

# Table promotions actives

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE TABLE ANALYTICS.PROMOTION_ACTIVES AS
SELECT
    promotion_id,
    product_category,
    promotion_type,
    discount_percentage,
    region,
    start_date,
    end_date,
    DATEDIFF(day, start_date, end_date) AS promotion_duration
FROM anycompany_lab.silver.promotions_data_clean;

In [ ]:
%%sql -r dataframe_7
select * from promotion_actives;

# Table clients enrichis

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE TABLE anycompany_lab.ANALYTICS.Client_enrichis AS
SELECT
    customer_id,
    name,
    gender,
    marital_status,
    annual_income,
    region,
    country,
    city,

    CASE
        WHEN annual_income < 30000 THEN 'Low Income'
        WHEN annual_income BETWEEN 30000 AND 70000 THEN 'Middle Income'
        ELSE 'High Income'
    END AS tranche_revenus,

    CASE
        WHEN DATEDIFF(YEAR, date_of_birth, CURRENT_DATE) < 25 THEN 'Under 25'
        WHEN DATEDIFF(YEAR, date_of_birth, CURRENT_DATE) BETWEEN 25 AND 40 THEN '25-40'
        WHEN DATEDIFF(YEAR, date_of_birth, CURRENT_DATE) BETWEEN 41 AND 60 THEN '41-60'
        ELSE '60+'
    END AS tranche_age

FROM anycompany_lab.silver.customer_demographics_clean;

In [ ]:
%%sql -r dataframe_9
select * from anycompany_lab.ANALYTICS.Client_enrichis;

# KPI

In [ ]:
%%sql -r dataframe_10
CREATE OR REPLACE VIEW ANYCOMPANY_LAB.ANALYTICS.SALES_KPI AS
SELECT
    region,
    year,
    month,
    SUM(amount) AS total_sales,
    ROUND(AVG(amount), 2) AS avg_sales,
    COUNT(transaction_id) AS number_of_sales,
    SUM(promotion_flag) AS sales_with_promotion,
    SUM(campaign_flag) AS sales_with_marketing
FROM ANYCOMPANY_LAB.ANALYTICS.vente_enrichies
GROUP BY region, year, month;

In [ ]:
%%sql -r dataframe_11
SELECT * FROM ANYCOMPANY_LAB.ANALYTICS.SALES_KPI;

# Features clients

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE TABLE ANYCOMPANY_LAB.ANALYTICS.CUSTOMER_FEATURES AS
SELECT
    c.customer_id,
    c.tranche_age,
    c.tranche_revenus,
    c.region,
    COUNT(s.transaction_id) AS nombre_d_achat,
    SUM(s.amount) AS total_depense,
    ROUND(AVG(s.amount), 2) AS panier_moyen,
    SUM(s.promotion_flag) AS promotion_usage
FROM ANYCOMPANY_LAB.ANALYTICS.client_enrichis c
LEFT JOIN ANYCOMPANY_LAB.ANALYTICS.vente_enrichies s
    ON c.region = s.region
GROUP BY
    c.customer_id,
    c.tranche_age,
    c.tranche_revenus,
    c.region;

In [ ]:
%%sql -r dataframe_12
select * from ANYCOMPANY_LAB.ANALYTICS.customer_features;